# 03a — Y2Y corridor-wide optimization

Runs prioritizr on the **full** aligned stack (no crop) — the original corridor-wide analysis,
now over the shared `prioritizr_core.R` engine. All parameters come from
`config.ANALYSES["y2y"]` (which mirrors the module defaults), carried via `manifest.json`
(cell 1 refreshes it). Outputs → `output_data/iter6_y2y/`.

**Configure a run:** edit `config.ANALYSES["y2y"]`, then run this notebook.
**Kernel:** `R (y2y)`. Run cell-by-cell; Ethan runs, Claude never executes.

In [ ]:
# ---- Setup: shared engine + this analysis' key + manifest refresh --------
source("prioritizr_core.R")            # pr_* functions (crop/mask, lock-in, weights, solve)
ANALYSIS <- "y2y"            # <-- the ONLY line that differs between 03a / 03b / 03c
PROJ <- normalizePath(getwd())         # run from the project root

mpath <- pr_refresh_manifest(PROJ, ANALYSIS)   # regenerate manifest.json from config for THIS
ctx   <- pr_setup(mpath, PROJ)                 # analysis (stops on failure); print the banner

In [ ]:
# ---- Ingest the stack + crop to the ROI + normalize (window set in config.ANALYSES) ----
ctx <- modifyList(ctx, pr_ingest(ctx))

In [ ]:
# ---- Planning units + lock-in + feasibility check ----
ctx <- modifyList(ctx, pr_planning_units(ctx))

In [ ]:
# ---- Feature weights (+ any per-analysis up-weighting) ----
ctx <- modifyList(ctx, pr_weights(ctx))

In [ ]:
# ---- Spatial-penalty matrices (built only for penalties > 0) ----
ctx <- modifyList(ctx, pr_penalty_matrices(ctx))

In [ ]:
# ---- Build the conservation problem ----
bp <- pr_build_problem(ctx); ctx$p <- bp$p; ctx$solve_params <- bp$solve_params

In [ ]:
# ---- Solve (Ethan runs; heavy) -- HiGHS single solution / Gurobi portfolio ----
# A run that hits the time limit returns an INFEASIBLE point (area > budget) -- discard it.
sv <- pr_solve(ctx); ctx$s <- sv$s; ctx$timing <- sv$timing; ctx$n_sol <- sv$n_sol

In [ ]:
# ---- Per-alternative summaries + selection-frequency map ----
ctx <- modifyList(ctx, pr_summaries(ctx))

In [ ]:
# ---- Write outputs for 04 (portfolio / frequency / representation / run_summary) ----
pr_write_outputs(ctx)